In [1]:
import warnings
import os
warnings.simplefilter(action='ignore')
os.environ["PYTHONWARNINGS"] = "ignore"

In [2]:
#parameters

### USER EDIT start
esm_file='/g/data/zv30/non-cmip/ACCESS-CM3/cm3-run-11-08-2025-25km-beta-om3-new-um-params/cm3-demo-datastore/cm3-demo-datastore.json'
# esm_file= os.path.join(run_dir, 'cm3-demo-datastore/cm3-virtual-datastore.json')
plotfolder='/g/data/tm70/ek4684/access-om3-paper-1/notebooks/mkfigs_output4/'
dpi=300
### USER EDIT stop

import matplotlib as mpl
import os
%matplotlib inline
mpl.rcParams['figure.dpi']= dpi

os.makedirs(plotfolder, exist_ok=True)

 # a similar cell under this means it's being run in batch
print("ESM datastore path: ",esm_file)
print("Plot folder path: ",plotfolder)

ESM datastore path:  /g/data/zv30/non-cmip/ACCESS-CM3/cm3-run-11-08-2025-25km-beta-om3-new-um-params/cm3-demo-datastore/cm3-demo-datastore.json
Plot folder path:  /g/data/tm70/ek4684/access-om3-paper-1/notebooks/mkfigs_output4/


In [3]:
import xarray as xr
import cf_xarray as cfxr
import intake
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
from distributed import Client
import numpy as np
import dask.array as da
import iris

In [4]:
client = Client(threads_per_worker=1)
print(client.dashboard_link)

/proxy/8787/status


INFO:fsspec.reference:Read reference from URL /g/data/zv30/non-cmip/ACCESS-CM3/cm3-run-11-08-2025-25km-beta-om3-new-um-params/cm3-demo-datastore/virtualised_outputs/seaIce.1mon.nbnd:2.nc:5.ni:1440.nj:1142.nkaer:5.nkbio:3.nkice:4.nksnow:1.json
INFO:fsspec.reference:Read reference from URL /g/data/zv30/non-cmip/ACCESS-CM3/cm3-run-11-08-2025-25km-beta-om3-new-um-params/cm3-demo-datastore/virtualised_outputs/seaIce.1mon.nbnd:2.nc:5.ni:1440.nj:1142.nkaer:5.nkbio:3.nkice:4.nksnow:1.json
INFO:fsspec.reference:Read reference from URL /g/data/zv30/non-cmip/ACCESS-CM3/cm3-run-11-08-2025-25km-beta-om3-new-um-params/cm3-demo-datastore/virtualised_outputs/atmos.1mon.bnds:2.depth:4.lat:144.lat_river:180.lat_v:145.lon:192.lon_river:360.lon_u:192.model_rho_level_number:85.model_theta_level_number:85.model_theta_level_number_0:50.model_theta_level_number_2:52.pressure:17.pseudo_level:6.pseudo_level_0:5.ps.json
INFO:fsspec.reference:Read reference from URL /g/data/zv30/non-cmip/ACCESS-CM3/cm3-run-11-08-

## Load datastore and datasets

In [5]:
#datastore_path = "/g/data/ol01/access-om3-output/access-om3-025/MC_25km_jra_ryf-1.0-beta/experiment_datastore.json"
COLUMNS_WITH_ITERABLES = [
        "variable",
        "variable_long_name",
        "variable_standard_name",
        "variable_cell_methods",
        "variable_units"
]

datastore = intake.open_esm_datastore(
    esm_file,
    columns_with_iterables=COLUMNS_WITH_ITERABLES
)

In [6]:
fn = "seaIce.1mon.nbnd:2.nc:5.ni:1440.nj:1142.nkaer:5.nkbio:3.nkice:4.nksnow:1.json"
fp = os.path.join(os.path.split(esm_file)[0], 'virtualised_outputs', fn)
cm3_ice_ds = xr.open_dataset(fp, engine="kerchunk")
cm3_ice_ds = cm3_ice_ds.rename(dict(ni='xh', nj='yh'))

INFO:fsspec.reference:Read reference from URL /g/data/zv30/non-cmip/ACCESS-CM3/cm3-run-11-08-2025-25km-beta-om3-new-um-params/cm3-demo-datastore/virtualised_outputs/seaIce.1mon.nbnd:2.nc:5.ni:1440.nj:1142.nkaer:5.nkbio:3.nkice:4.nksnow:1.json
INFO:fsspec.reference:Read reference from URL /g/data/zv30/non-cmip/ACCESS-CM3/cm3-run-11-08-2025-25km-beta-om3-new-um-params/cm3-demo-datastore/virtualised_outputs/seaIce.1mon.nbnd:2.nc:5.ni:1440.nj:1142.nkaer:5.nkbio:3.nkice:4.nksnow:1.json
INFO:fsspec.reference:Read reference from URL /g/data/zv30/non-cmip/ACCESS-CM3/cm3-run-11-08-2025-25km-beta-om3-new-um-params/cm3-demo-datastore/virtualised_outputs/seaIce.1mon.nbnd:2.nc:5.ni:1440.nj:1142.nkaer:5.nkbio:3.nkice:4.nksnow:1.json


In [7]:
fn = "atmos.1mon.bnds:2.depth:4.lat:144.lat_river:180.lat_v:145.lon:192.lon_river:360.lon_u:192.model_rho_level_number:85.model_theta_level_number:85.model_theta_level_number_0:50.model_theta_level_number_2:52.pressure:17.pseudo_level:6.pseudo_level_0:5.ps.json"
fp = os.path.join(os.path.split(esm_file)[0], 'virtualised_outputs', fn)
cm3_atm_ds = xr.open_dataset(fp, engine="kerchunk")

INFO:fsspec.reference:Read reference from URL /g/data/zv30/non-cmip/ACCESS-CM3/cm3-run-11-08-2025-25km-beta-om3-new-um-params/cm3-demo-datastore/virtualised_outputs/atmos.1mon.bnds:2.depth:4.lat:144.lat_river:180.lat_v:145.lon:192.lon_river:360.lon_u:192.model_rho_level_number:85.model_theta_level_number:85.model_theta_level_number_0:50.model_theta_level_number_2:52.pressure:17.pseudo_level:6.pseudo_level_0:5.ps.json
INFO:fsspec.reference:Read reference from URL /g/data/zv30/non-cmip/ACCESS-CM3/cm3-run-11-08-2025-25km-beta-om3-new-um-params/cm3-demo-datastore/virtualised_outputs/atmos.1mon.bnds:2.depth:4.lat:144.lat_river:180.lat_v:145.lon:192.lon_river:360.lon_u:192.model_rho_level_number:85.model_theta_level_number:85.model_theta_level_number_0:50.model_theta_level_number_2:52.pressure:17.pseudo_level:6.pseudo_level_0:5.ps.json
INFO:fsspec.reference:Read reference from URL /g/data/zv30/non-cmip/ACCESS-CM3/cm3-run-11-08-2025-25km-beta-om3-new-um-params/cm3-demo-datastore/virtualised_o

## Load areas

In [8]:
wet = datastore.search(variable="wet").to_dask().compute()
areacello = datastore.search(variable="areacello").to_dask().compute()

areacello = (areacello.areacello * (wet.wet == 1.0))

In [9]:
EARTH_RADIUS = 6371229.0
nlon, nlat = 192, 144
dx = 360 / nlon
dy = 180 / nlat

element_lat = (np.arange(nlat) + 0.5) * dy  - 90
element_lat = element_lat[:, None] * np.ones(nlon)[None, :]
element_lon = (np.arange(nlon) + 0.5) * dx

pi_over_180 = np.pi / 180
element_areas = dx * pi_over_180 * (
  np.sin((element_lat + 0.5 * dy) * pi_over_180) - np.sin((element_lat - 0.5 * dy) * pi_over_180)
)

areacella = element_areas * EARTH_RADIUS**2
areacella = xr.DataArray(areacella, coords=dict(lat=element_lat[:, 0], lon=element_lon), dims=('lat', 'lon'))

In [10]:
ancil_dir = '/scratch/tm70/kr4383/cylc-run/ancil-gen-30-07-2025/share/data/n96e_mom025_20250515/'
land_frac = iris.load_cube(os.path.join(ancil_dir, 'qrparm.landfrac')).data
vsat = iris.load_cube(os.path.join(ancil_dir, 'qrparm.soil'), 'soil_porosity').data

land_area = land_frac * areacella
vsat_area = vsat.data * land_area

## Load different mass components

In [11]:
ssh = datastore.search(variable="SSH", frequency="1mon").to_dask(
    xarray_open_kwargs = dict(
        chunks={"yh": -1, "xh": -1}, # Good for spatial operations, but not temporal
        decode_timedelta=True
    )
)

ocn_column_mass = datastore.search(variable="mass_wt", frequency="1mon").to_dask(
    xarray_open_kwargs = dict(
        chunks={"yh": -1, "xh": -1}, # Good for spatial operations, but not temporal
        decode_timedelta=True
    )
)

In [12]:
%%time
avg_ssh = ssh.weighted(areacello.fillna(0)).mean(dim=('yh', 'xh')).compute()
ocn_mass = ocn_column_mass.weighted(areacello.fillna(0)).sum(dim=('yh', 'xh')).compute()

CPU times: user 1.74 s, sys: 550 ms, total: 2.29 s
Wall time: 25.6 s


In [13]:
%%time
sea_ice_vol = cm3_ice_ds.hi_m.weighted(areacello.fillna(0)).sum(dim=('yh', 'xh')).compute()
sea_snow_vol = cm3_ice_ds.hs_m.weighted(areacello.fillna(0)).sum(dim=('yh', 'xh')).compute()

INFO:fsspec.reference:Read reference from URL /g/data/zv30/non-cmip/ACCESS-CM3/cm3-run-11-08-2025-25km-beta-om3-new-um-params/cm3-demo-datastore/virtualised_outputs/seaIce.1mon.nbnd:2.nc:5.ni:1440.nj:1142.nkaer:5.nkbio:3.nkice:4.nksnow:1.json
INFO:fsspec.reference:Read reference from URL /g/data/zv30/non-cmip/ACCESS-CM3/cm3-run-11-08-2025-25km-beta-om3-new-um-params/cm3-demo-datastore/virtualised_outputs/seaIce.1mon.nbnd:2.nc:5.ni:1440.nj:1142.nkaer:5.nkbio:3.nkice:4.nksnow:1.json
INFO:fsspec.reference:Read reference from URL /g/data/zv30/non-cmip/ACCESS-CM3/cm3-run-11-08-2025-25km-beta-om3-new-um-params/cm3-demo-datastore/virtualised_outputs/seaIce.1mon.nbnd:2.nc:5.ni:1440.nj:1142.nkaer:5.nkbio:3.nkice:4.nksnow:1.json
INFO:fsspec.reference:Read reference from URL /g/data/zv30/non-cmip/ACCESS-CM3/cm3-run-11-08-2025-25km-beta-om3-new-um-params/cm3-demo-datastore/virtualised_outputs/seaIce.1mon.nbnd:2.nc:5.ni:1440.nj:1142.nkaer:5.nkbio:3.nkice:4.nksnow:1.json


CPU times: user 10.1 s, sys: 2.76 s, total: 12.9 s
Wall time: 2min 2s


In [ ]:
%%time
atm_total_mass = cm3_atm_ds.fld_s30i404.weighted(areacella).sum(dim=('lat', 'lon')).compute()
atm_dry_mass = cm3_atm_ds.fld_s30i403.weighted(areacella).sum(dim=('lat', 'lon')).compute()
atm_water_mass = atm_total_mass - atm_dry_mass

land_snow_mass = cm3_atm_ds.fld_s08i023.weighted(land_area).sum(dim=('lat', 'lon')).compute()
soil_water_mass = cm3_atm_ds.fld_s08i223.weighted(land_area).sum(dim=('lat', 'lon', 'depth')).compute()
river_mass = cm3_atm_ds.fld_s26i001.sum(('lat_river', 'lon_river')).compute()
ground_water_mass = 3000 * cm3_atm_ds.fld_s08i250.weighted(vsat_area).sum(dim=('lat', 'lon')).compute()

In [ ]:
soil_water_mass = cm3_atm_ds.fld_s08i223.weighted(land_area).sum(dim=('lat', 'lon', 'depth')).compute()
ground_water_mass = 3000 * cm3_atm_ds.fld_s08i250.weighted(vsat_area).sum(dim=('lat', 'lon')).compute()

In [16]:
# CICE ice and snow densities
rhoi = 917.
rhos = 330.
ocean_density = 1026.

In [ ]:
sea_ice_mass = rhoi * sea_ice_vol + rhos * sea_snow_vol
atm_mass = atm_water_mass + land_snow_mass + soil_water_mass + river_mass + ground_water_mass

In [ ]:
total_mass = sea_ice_mass[:480] + atm_mass[:480] + ocn_mass.mass_wt[:480]

In [ ]:
mass_error_mm = (total_mass - total_mass[0]) * 1000 / (ocean_density * areacello.sum())
mass_error_mm.plot()
plt.title('Water mass error (mm)')

In [ ]:
relative_mass_error = (total_mass - total_mass[0]) / total_mass[0]
relative_mass_error.plot()
plt.title('Relative water mass error')